In [1]:
import pandas as pd
import numpy as np
import torch
import time
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.metrics import classification_report, confusion_matrix
from transformers import (
    DistilBertTokenizerFast,
    DistilBertForSequenceClassification,
    Trainer,
    TrainingArguments,
    EarlyStoppingCallback
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


Using device: cuda


In [2]:
train_df = pd.read_csv("../data/processed/train_all_versions.csv")
val_df = pd.read_csv("../data/processed/val_all_versions.csv")
test_df = pd.read_csv("../data/processed/test_all_versions.csv")


In [3]:
binary_mapping = {
    "very_negative": 0,
    "negative": 0,
    "neutral": 0,
    "positive": 1,
    "very_positive": 1
}

for df in [train_df, val_df, test_df]:
    df["label"] = df["sentiment"].map(binary_mapping)
    df.dropna(subset=["text_dl", "label"], inplace=True)


In [4]:
train_texts = train_df["text_dl"].astype(str).tolist()
val_texts = val_df["text_dl"].astype(str).tolist()
test_texts = test_df["text_dl"].astype(str).tolist()

train_labels = train_df["label"].values
val_labels = val_df["label"].values
test_labels = test_df["label"].values


Tokenizer replaces padding and vocab

In [5]:
tokenizer = DistilBertTokenizerFast.from_pretrained("distilbert-base-uncased")

max_len = 150   

train_encodings = tokenizer(train_texts, truncation=True, padding=True, max_length=max_len)
val_encodings = tokenizer(val_texts, truncation=True, padding=True, max_length=max_len)
test_encodings = tokenizer(test_texts, truncation=True, padding=True, max_length=max_len)


'[Errno 11001] getaddrinfo failed' thrown while requesting HEAD https://huggingface.co/distilbert-base-uncased/resolve/main/tokenizer_config.json
Retrying in 1s [Retry 1/5].


RuntimeError: Cannot send a request, as the client has been closed.

Dataset class 

In [ ]:
class SentimentDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)

train_dataset = SentimentDataset(train_encodings, train_labels)
val_dataset = SentimentDataset(val_encodings, val_labels)
test_dataset = SentimentDataset(test_encodings, test_labels)


In [ ]:
class_counts = train_df["label"].value_counts().sort_index()
total = sum(class_counts)
weights = [total / (2 * count) for count in class_counts]
class_weights = torch.tensor(weights).to(device)


Load model 


In [ ]:
model = DistilBertForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=2
)

model.to(device)


CUstom trainer 

In [ ]:
from torch.nn import CrossEntropyLoss

class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False):
        labels = inputs.get("labels")
        outputs = model(**inputs)
        logits = outputs.get("logits")

        loss_fct = CrossEntropyLoss(weight=class_weights, label_smoothing=0.05)
        loss = loss_fct(logits, labels)

        return (loss, outputs) if return_outputs else loss


In [ ]:
training_args = TrainingArguments(
    output_dir="./results",

    # Training
    num_train_epochs=4,                  
    per_device_train_batch_size=8,      
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=4,       

    # Optimization
    learning_rate=2e-5,
    weight_decay=0.01,
    warmup_ratio=0.1,

    # GPU Optimization
    fp16=True,                         
    dataloader_num_workers=2,

    # Evaluation
    evaluation_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",

    # Logging
    logging_dir="./logs",
    logging_steps=200,
    report_to="none"
)


In [ ]:
trainer = WeightedTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
)


In [ ]:
import time

start = time.time()
trainer.train()
end = time.time()

print(f"Training time: {(end-start)/60:.2f} minutes")


Evaluation 


In [ ]:
predictions = trainer.predict(test_dataset)
y_pred = np.argmax(predictions.predictions, axis=1)

print(classification_report(test_labels, y_pred))

cm = confusion_matrix(test_labels, y_pred)

plt.figure(figsize=(6,5))
sns.heatmap(cm, annot=True, fmt="d",
            xticklabels=["Negative", "Positive"],
            yticklabels=["Negative", "Positive"],
            cmap="magma")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Confusion Matrix - DistilBERT")
plt.show()


Prediction

In [ ]:
def predict_review(sentence):
    model.eval()
    
    inputs = tokenizer(
        sentence,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=max_len
    ).to(device)

    with torch.no_grad():
        outputs = model(**inputs)
        probs = torch.softmax(outputs.logits, dim=1)
        confidence, prediction = torch.max(probs, dim=1)

    label = "Positive" if prediction.item() == 1 else "Negative"
    
    return label, confidence.item()
